# Przygotowanie i czyszczenie danych (Data Cleaning)

Celem tego notebooka jest przygotowanie danych sprzedażowych do dalszej analizy.

Dataset pochodzi z publicznego zbioru danych e-commerce (Olist) i zawiera informacje o:

- zamówieniach
- produktach
- klientach
- płatnościach
- sprzedawcach

Na tym etapie wykonujemy:

- wczytanie danych
- wstępną eksplorację
- analizę brakujących wartości
- czyszczenie danych
- połączenie tabel w jeden zbiór analityczny

Efektem końcowym będzie przygotowanie datasetu, który zostanie wykorzystany w dalszej analizie eksploracyjnej (EDA).

## Import bibliotek

Na początku importujemy biblioteki wykorzystywane do analizy danych.

W projekcie korzystamy głównie z:

- pandas — manipulacja danymi
- numpy — operacje numeryczne

In [2]:
import pandas as pd
import numpy as np

## Wczytanie danych

Dataset Olist składa się z kilku tabel. Najważniejsze z nich to:

- **orders** — informacje o zamówieniach
- **order_items** — produkty w zamówieniach
- **products** — informacje o produktach
- **customers** — informacje o klientach
- **payments** — informacje o płatnościach

Każda z tych tabel zostanie wczytana osobno.

In [3]:
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

## Wstępna eksploracja danych

Sprawdzamy podstawowe informacje o datasetach:

- liczbę rekordów
- typy kolumn
- przykładowe dane

In [4]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [6]:
order_items.head()
products.head()
customers.head()
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


## Wstępna eksploracja danych

Na tym etapie sprawdzamy podstawowe informacje o danych:

- liczbę rekordów
- typy kolumn
- przykładowe wartości
- strukturę tabel

Pozwala to lepiej zrozumieć dataset przed dalszą analizą.

In [7]:
def inspect_df(df, name):
    print("DATASET:", name)
    print("------------------")
    print(df.shape)
    print(df.isnull().sum())
    print(df.dtypes)
    print(df.head())

In [8]:
inspect_df(orders, "orders")
inspect_df(order_items, "order_items")
inspect_df(products, "products")
inspect_df(customers, "customers")
inspect_df(payments, "payments")

DATASET: orders
------------------
(99441, 8)
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44

## Konwersja dat

In [9]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

## Łączenie tabel

Dataset e-commerce jest podzielony na kilka tabel.

Aby przeprowadzić analizę sprzedaży, konieczne jest ich połączenie w jeden zbiór danych.

W tym celu wykorzystujemy operacje **merge** w bibliotece pandas.

Połączone zostaną informacje o:

- zamówieniach
- produktach
- klientach
- płatnościach

In [10]:
df = order_items.merge(
    orders,
    on="order_id",
    how="left"
)

In [11]:
df = df.merge(
    products,
    on="product_id",
    how="left"
)

In [12]:
df = df.merge(
    customers,
    on="customer_id",
    how="left"
)

In [13]:
df = df.merge(
    payments,
    on="order_id",
    how="left"
)

## Dodanie kolumny revenue

In [14]:
df["revenue"] = df["price"] + df["freight_value"]

## Sprawdzenie duplikatów

In [15]:
duplicates = df.duplicated().sum()
print("Number of duplicated rows:", duplicates)

Number of duplicated rows: 0


## Analiza brakujących wartości

Sprawdzamy liczbę brakujących wartości w każdej kolumnie.

Brakujące dane są częstym problemem w rzeczywistych datasetach i mogą wpływać na wyniki analizy.

Na podstawie tej analizy zdecydujemy:

- które wartości można uzupełnić
- które rekordy należy usunąć
- które brakujące dane są naturalne i mogą pozostać

In [16]:
df.isnull().sum()

order_id                            0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  15
order_delivered_carrier_date     1245
order_delivered_customer_date    2567
order_estimated_delivery_date       0
product_category_name            1698
product_name_lenght              1698
product_description_lenght       1698
product_photos_qty               1698
product_weight_g                   20
product_length_cm                  20
product_height_cm                  20
product_width_cm                   20
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
payment_sequ

### Interpretacja brakujących danych

W datasetcie występują brakujące wartości w kilku kolumnach, m.in.:

- informacje o kategorii produktu
- wymiary produktów
- dane dotyczące płatności

W dalszym kroku dane te zostaną odpowiednio przetworzone.

## Czyszczenie danych

Na podstawie wcześniejszej analizy brakujących danych wykonujemy:

- uzupełnienie brakujących kategorii produktów
- usunięcie rekordów z brakującymi informacjami o płatności

In [17]:
df["product_category_name"] = df["product_category_name"].fillna("unknown")

In [18]:
df = df.dropna(subset=[
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
])

In [19]:
df = df.dropna(subset=["payment_value"])

### Analiza brakujących wartości

Zbiór danych zawiera brakujące wartości w kilku kolumnach:

- informacje o kategorii produktu (~1,7 tys. rekordów)
- wymiary produktu (20 rekordów)
- daty dostawy niektórych zamówień
- informacje o płatnościach (3 rekordy)

Strategia postępowania:

- brakujące kategorie produktów zastąpiono wartością „unknown”
- niekompletne rekordy płatności usunięto
- daty dostawy pozostawiono bez zmian, ponieważ odzwierciedlają one rzeczywisty stan zamówień


In [20]:
df["order_month"] = df["order_purchase_timestamp"].dt.to_period("M")

df["order_year"] = df["order_purchase_timestamp"].dt.year

In [21]:
df["order_value"] = df.groupby("order_id")["revenue"].transform("sum")

## Zapis przetworzonych danych

Oczyszczony dataset zapisujemy w folderze `data/processed`, aby mógł zostać wykorzystany w kolejnych etapach analizy.

In [22]:
df.to_csv("../data/processed/sales_cleaned.csv", index=False)